## Setup

### General

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd

from _utilities.stupidloggen import StupidLogGenerator


### Paths & Logger

In [2]:
ROOT_DIR = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(ROOT_DIR)

PATH_ORG_DATA = "data/compas-scores-two-years.csv"
PATH_SYNTHETIC_DATA = "data/synthetic/"
PATH_FOLDER_MODELS = "models/"

if not os.path.exists(PATH_FOLDER_MODELS):
    os.makedirs(PATH_FOLDER_MODELS)
if not os.path.exists(PATH_SYNTHETIC_DATA):
    os.makedirs(PATH_SYNTHETIC_DATA)

In [3]:
logger = StupidLogGenerator().initLogger()

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- is_recid

In [4]:
scores_two_years = pd.read_csv(PATH_ORG_DATA)

columns_to_include = [
    "sex",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
    "is_recid",
]

dataset = scores_two_years[columns_to_include].copy()

In [5]:
categorical_columns = [
    "sex",
    "age_cat",
    "race",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
    "is_recid",
]

In [6]:
dataset.nunique()

sex                  2
age                 65
age_cat              3
race                 6
juv_fel_count       11
juv_misd_count      10
juv_other_count     10
priors_count        37
c_charge_degree      2
c_charge_desc      437
r_charge_degree     10
r_charge_desc      340
two_year_recid       2
is_recid             2
dtype: int64

## Data Preprocessing

When `is_recid` is 0 there should not be any values in `r_charge_degree` and `r_charge_desc`, as they contain information about the crime they were re-arested for. Thus, we made a separate category "Not applicable" to make it clear which values are truly missing.

In [7]:
recid_charge_cols = [
    "r_charge_degree",
    "r_charge_desc",
]

for col in recid_charge_cols:
    dataset[col] = np.where((dataset["is_recid"] == 0), "Not applicable", dataset[col])

The `race` attribute is distributed as indicated in the table below. The ammount of data on "Hispanic", "Asian", and "Native American" individually is not enough to be significant, therefore we combine these together with "Other".

| Race             | Count | Percent |
| --- | --- | ---  |
| African-American |  3696 |  51.23% |
| Caucasian        |  2454 |  34.02% |
| Hispanic         |   637 |   8.83% |
| Other            |   377 |   5.23% |
| Asian            |    32 |   0.44% |
| Native American  |    18 |   0.25% |

In [8]:
other_races = ["Hispanic", "Asian", "Native American"]

dataset["race"] = dataset["race"].replace(other_races, "Other")

In [9]:
dataset.to_csv("data/preprocessed_real_data.csv", index=False)

## Model Training

#### Model Setup

In [10]:
# To make sure that FairGAN can be imported
path_to_dir = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(path_to_dir)

In [11]:
import pickle

from ctgan import CTGAN, TVAE
from tabfairgan import TFG

from fairgan import FairGAN


In [12]:
def save_successful(
    model, model_name: str, model_variant: str, synthetic_data: pd.DataFrame
):
    try:
        data_folder = PATH_SYNTHETIC_DATA + f"/{model_variant}"
        model_folder = PATH_FOLDER_MODELS + f"/{model_variant}"

        os.makedirs(data_folder, exist_ok=True)
        os.makedirs(model_folder, exist_ok=True)
        synthetic_data.to_csv(
            f"{data_folder}/{model_name}_synthetic.csv",
            index=False,
        )

        with open(f"{model_folder}/{model_name}_model.pkl", "wb") as f:
            pickle.dump(model, f)

        logger.info(f"Saved {model_name} : {model_variant}")
    except Exception as e:
        logger.error(e)

#### CTGAN

The CTGAN model learns the joint distribution of the dataset.

In [13]:
def run_CTGAN(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    verbose: bool,
    synthetic_data_count: int,
    dataset_variant: str,
):
    logger.info("\n\nStarting to train CTGAN model")

    try:
        # Train CTGAN model
        ctgan_df = df.copy()

        ctgan_model = CTGAN(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
        ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

        # Generate synthetic data using CTGAN
        ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)
        logger.info(f"Completed CTGAN : {dataset_variant}")

        save_successful(ctgan_model, "ctgan", dataset_variant, ctgan_synthetic_data)

    except Exception as e:
        logger.error(e)

#### TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [14]:
def run_TVAE(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    verbose: bool,
    synthetic_data_count: int,
    dataset_variant: str,
):
    logger.info("\n\nStarting to train TVAE model\n\n")

    try:
        # Train TVAE model
        tvae_df = df.copy()

        tvae_model = TVAE(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
        tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

        # Generate synthetic data using TVAE
        tvae_synthetic_data = tvae_model.sample(synthetic_data_count)
        logger.info(f"Completed TVAE : {dataset_variant}")

        save_successful(tvae_model, "tvae", dataset_variant, tvae_synthetic_data)

    except Exception as e:
        logger.error(e)

#### FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

In [15]:
def run_FairGAN(
    df: pd.DataFrame,
    num_epochs: int,
    synthetic_data_count: int,
    sensitive_feature: str,
    target_feature: str,
    dataset_variant: str,
):
    logger.info("\n\nStarting to train FairGAN with different lambda_fair values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    fairgan_datasets = {}
    fairgan_models = {}

    for lam in lambda_values:
        logger.info(f"\nTraining FairGAN with lambda_fair = {lam}")

        fairgan_df = df.copy()

        try:
            # Initialize model
            fairgan_model = FairGAN(
                df=fairgan_df,
                sensitive_feature=sensitive_feature,
                target_feature=target_feature,
                epochs=num_epochs,
                noise_dim=100,
                lr=0.001,
                verbose=True,
            )

            # Train model
            fairgan_model.train(lambda_fair=lam)

            # Generate synthetic dataset
            synthetic_df = fairgan_model.generate(synthetic_data_count)

            dataset_name = f"FairGAN_lambda_{lam}"
            fairgan_datasets[dataset_name] = synthetic_df
            fairgan_models[dataset_name] = fairgan_model
            logger.info(f"Completed FairGAN with lambda_fair = {lam}")

            save_successful(fairgan_model, dataset_name, dataset_variant, synthetic_df)

        except Exception as e:
            logger.error(e)

#### TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

**Fairness Constraints**

To generate fair synthetic data, you need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [16]:
# TODO: Define proper Y_desire


def run_TabFairGAN(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    synthetic_data_count: int,
    fairness_config: dict,
    dataset_variant: str,
):
    logger.info("\n\nStarting to train TabFairGAN with different lambda values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    tabfairgan_datasets = {}
    tabfairgan_models = {}

    for lam in lambda_values:
        if lam == 0.0:
            fairness_config["fair_epochs"] = 0
        else:
            fairness_config["fair_epochs"] = 50

        fairness_config["lamda"] = lam

        try:
            logger.info(f"\nTraining TabFairGAN with lambda = {lam}")

            tfg_df = df.copy()

            # Turn the sensitive and target attribute into categorical string
            tfg_df["sex"] = tfg_df["sex"].astype(str)
            tfg_df["is_recid"] = tfg_df["is_recid"].astype(str)

            # Initialize model
            tfg_model = TFG(
                tfg_df,
                epochs=num_epochs,
                batch_size=batch_size,
                fairness_config=fairness_config,
            )

            # Train model
            tfg_model.train()

            # Generate synthetic data
            tfg_synthetic_data = tfg_model.generate_fake_df(
                num_rows=synthetic_data_count
            )

            dataset_name = f"TabFairGAN_lambda_{lam}"
            tabfairgan_datasets[dataset_name] = tfg_synthetic_data
            tabfairgan_models[dataset_name] = tfg_model
            logger.info(f"Completed {dataset_name} : {dataset_variant}")
            save_successful(
                tfg_model, dataset_name, dataset_variant, tfg_synthetic_data
            )

            logger.info(f"Completed {dataset_name}")

        except Exception as e:
            logger.error(e)
            continue

## Model Settings

In [17]:
# Set logging level
logger.setLevel(logging.INFO)

In [18]:
target_feature = "is_recid"
sensitive_feature = "sex"
sensitive_under = "Female"

fairness_config = {
    "S": sensitive_feature,
    "Y": target_feature,
    "S_under": sensitive_under,
    "Y_desire": "0",
}

In [19]:
verbose = 2
synthetic_data_count = 100
num_epochs = 10
batch_size = 100

test_dataset = dataset.sample(n=500)

In [20]:
carge_desc_cols = ["c_charge_desc", "r_charge_desc"]
age_cat_cols = ["age_cat"]

dataset_versions = [
    [dataset, "Original"],
    [dataset.drop(columns=carge_desc_cols), "With_age_cat No_carge_desc"],
    [dataset.drop(columns=age_cat_cols + carge_desc_cols), "No_age_cat No_carge_desc"],
]

## Run models

In [21]:
for dataset_version in dataset_versions:
    logger.info(f"\n\nRunning models for dataset version: {dataset_version[1]}")
    run_CTGAN(
        dataset_version[0],
        num_epochs,
        batch_size,
        verbose,
        synthetic_data_count,
        dataset_version[1],
    )
    run_TVAE(
        dataset_version[0],
        num_epochs,
        batch_size,
        verbose,
        synthetic_data_count,
        dataset_version[1],
    )
    run_FairGAN(
        dataset_version[0],
        num_epochs,
        synthetic_data_count,
        sensitive_feature,
        target_feature,
        dataset_version[1],
    )
    run_TabFairGAN(
        dataset_version[0],
        num_epochs,
        batch_size,
        synthetic_data_count,
        fairness_config,
        dataset_version[1],
    )

[INFO]: 

Running models for dataset version: Original
[INFO]: 

Starting to train CTGAN model
Gen. (+01.37) | Discrim. (+00.03): 100%|██████████| 10/10 [01:01<00:00,  6.15s/it]
[INFO]: Completed CTGAN : Original
[INFO]: Saved ctgan : Original
[INFO]: 

Starting to train TVAE model


Loss: +11.47: 100%|██████████| 10/10 [00:05<00:00,  1.78it/s]
[INFO]: Completed TVAE : Original
[INFO]: Saved tvae : Original
[INFO]: 

Starting to train FairGAN with different lambda_fair values
[INFO]: 
Training FairGAN with lambda_fair = 0.0


[Pretrain] epoch 0 | loss 8700.9131
[Pretrain] epoch 5 | loss 6900.1172
[Pretrain] epoch 10 | loss 4411.2002
[Pretrain] epoch 15 | loss 1419.9036
Epoch 0 | D1: 0.6856 | D2: 1.3889 | G: 0.7049 | Fair: 0.6943


[INFO]: Completed FairGAN with lambda_fair = 0.0
[INFO]: Saved FairGAN_lambda_0.0 : Original
[INFO]: 
Training FairGAN with lambda_fair = 0.1


[Pretrain] epoch 0 | loss 8965.9512
[Pretrain] epoch 5 | loss 7029.9106
[Pretrain] epoch 10 | loss 4855.1255
[Pretrain] epoch 15 | loss 2078.4954
Epoch 0 | D1: 2.9676 | D2: 1.3846 | G: 0.6417 | Fair: 0.6937


[INFO]: Completed FairGAN with lambda_fair = 0.1
[INFO]: Saved FairGAN_lambda_0.1 : Original
[INFO]: 
Training FairGAN with lambda_fair = 0.5


...

epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10


[INFO]: Completed TabFairGAN_lambda_0.7 : No_age_cat No_carge_desc
[INFO]: Saved TabFairGAN_lambda_0.7 : No_age_cat No_carge_desc
[INFO]: Completed TabFairGAN_lambda_0.7
[INFO]: 
Training TabFairGAN with lambda = 1.0


Training Progress:   0%|                                                     | 0/10 [00:00<?, ?it/s]

epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10


[INFO]: Completed TabFairGAN_lambda_1.0 : No_age_cat No_carge_desc
[INFO]: Saved TabFairGAN_lambda_1.0 : No_age_cat No_carge_desc
[INFO]: Completed TabFairGAN_lambda_1.0
